In [19]:
# Import the required libraries.
import os

# Need to be installed.
import yaml
import joblib
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.model_selection import train_test_split

## **1 - Configuration File**
---

In [21]:
 # Function to load configuration parameter.
def load_config(path_config):
    """
    Load the configuration file (config.yaml).

    Parameters:
    ----------
    path_config : str
        Configuration file location.

    Returns:
    -------
    params : dict
        The configuration parameters.
    """

    # Try to load config.yaml file.
    try:
        with open(path_config, 'r') as file:
            params = yaml.safe_load(file)
    except FileNotFoundError as err:
        raise RuntimeError(f"Configuration file not found in {path_config}")

    return params

In [22]:
def update_config(key, value, params, path_config):
    """
    Update the configuration parameter values.

    Parameters:
    ----------
    key : str
        The key to be updated.

    value : any type supported in Python
        The updated value.

    params : dict
        Loaded configuration parameters.

    path_config : str
        Configuration file location.

    Returns:
    -------
    config : dict
        Updated configuration parameters.
    """

    # To maintain the raw config immutable.
    params = params.copy()

    # Update the configuration parameters.
    params[key] = value

    with open(path_config, 'w') as file:
        yaml.dump(params, file)

    print(f"Params Updated! \nKey: {key} \nValue: {value}\n")

    # Reload the updated configuration parameters.
    config = load_config(path_config)

    return config

In [23]:
# Load the configuration file.
PATH_CONFIG = "../config/config.yaml"
config = load_config(PATH_CONFIG)

In [24]:
config

{'datetime_columns': ['tanggal'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'int32_columns': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'label': 'categori',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'object_columns': ['stasiun', 'critical', 'categori'],
 'path_raw_data': '../data/raw/',
 'range_co': [0, 100],
 'range_no2': [0, 100],
 'range_o3': [0, 140],
 'range_pm10': [0, 800],
 'range_pm25': [0, 400],
 'range_so2': [0, 500],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4 (Lubang Buaya)',
  'DKI5 (Kebon Jeruk) Jakarta Barat']}

## **2 - Data Collection**
---


In [25]:
# Function to load data.
def load_data(path_data):
    """
    Load csv files and join into one dataframe.

    Parameters:
    ----------
    path_data : str
        Raw dataset location.

    Returns:
    -------
    raw_dataset : pd.DataFrame
        Loaded and joined data.
    """

    # Create variable to store raw dataset.
    raw_dataset = pd.DataFrame()

    # Load and join the csv files.
    for i in tqdm(os.listdir(path_data)):
        raw_dataset = pd.concat([pd.read_csv(path_data + i), raw_dataset])

    return raw_dataset

In [26]:
# Load the raw dataset.
PATH_RAW_DATA = config["path_raw_data"]
raw_dataset = load_data(PATH_RAW_DATA)

100%|█████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 76.23it/s]


In [29]:
raw_dataset

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
0,2021-02-01,DKI1 (Bunderan HI),59,79,21,26,15,31,79,PM25,SEDANG
1,2021-02-02,DKI1 (Bunderan HI),30,46,21,14,16,24,46,PM25,BAIK
2,2021-02-03,DKI1 (Bunderan HI),23,33,19,11,14,19,33,PM25,BAIK
3,2021-02-04,DKI1 (Bunderan HI),36,53,23,14,12,24,53,PM25,SEDANG
4,2021-02-05,DKI1 (Bunderan HI),29,36,20,14,12,21,36,PM25,BAIK
...,...,...,...,...,...,...,...,...,...,...,...
150,2021-10-27,DKI5 (Kebon Jeruk) Jakarta Barat,53,76,---,11,32,17,76,PM25,SEDANG
151,2021-10-28,DKI5 (Kebon Jeruk) Jakarta Barat,51,77,---,16,29,27,77,PM25,SEDANG
152,2021-10-29,DKI5 (Kebon Jeruk) Jakarta Barat,50,70,---,18,23,33,70,PM25,SEDANG
153,2021-10-30,DKI5 (Kebon Jeruk) Jakarta Barat,50,74,---,12,30,24,74,PM25,SEDANG


In [30]:
raw_dataset = raw_dataset.reset_index(drop=True)

In [31]:
raw_dataset

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
0,2021-02-01,DKI1 (Bunderan HI),59,79,21,26,15,31,79,PM25,SEDANG
1,2021-02-02,DKI1 (Bunderan HI),30,46,21,14,16,24,46,PM25,BAIK
2,2021-02-03,DKI1 (Bunderan HI),23,33,19,11,14,19,33,PM25,BAIK
3,2021-02-04,DKI1 (Bunderan HI),36,53,23,14,12,24,53,PM25,SEDANG
4,2021-02-05,DKI1 (Bunderan HI),29,36,20,14,12,21,36,PM25,BAIK
...,...,...,...,...,...,...,...,...,...,...,...
1825,2021-10-27,DKI5 (Kebon Jeruk) Jakarta Barat,53,76,---,11,32,17,76,PM25,SEDANG
1826,2021-10-28,DKI5 (Kebon Jeruk) Jakarta Barat,51,77,---,16,29,27,77,PM25,SEDANG
1827,2021-10-29,DKI5 (Kebon Jeruk) Jakarta Barat,50,70,---,18,23,33,70,PM25,SEDANG
1828,2021-10-30,DKI5 (Kebon Jeruk) Jakarta Barat,50,74,---,12,30,24,74,PM25,SEDANG


In [32]:
PATH_JOINED_DATA = f"../data/interim/joined_dataset.pkl"
joblib.dump(raw_dataset, PATH_JOINED_DATA)

['../data/interim/joined_dataset.pkl']

In [33]:
# Update the configuration parameter.
config = update_config(
    key = "path_joined_data",
    value = PATH_JOINED_DATA,
    params = config,
    path_config = PATH_CONFIG
)

Params Updated! 
Key: path_joined_data 
Value: ../data/interim/joined_dataset.pkl



## **3 - Data Validation**
---

In [34]:
raw_dataset.dtypes

tanggal        str
stasiun        str
pm10        object
pm25        object
so2         object
co          object
o3          object
no2         object
max         object
critical       str
categori       str
dtype: object

In [35]:
raw_dataset["tanggal"] = pd.to_datetime(raw_dataset["tanggal"])

In [36]:
raw_dataset["pm10"] = raw_dataset["pm10"].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [37]:
raw_dataset.eq("-1").any() | raw_dataset.eq(-1).any()

tanggal     False
stasiun     False
pm10        False
pm25        False
so2         False
co          False
o3          False
no2         False
max         False
critical    False
categori    False
dtype: bool

In [38]:
raw_dataset["pm10"] = raw_dataset["pm10"].replace("---", -1).astype(int)

In [39]:
raw_dataset["pm25"] = raw_dataset["pm25"].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [40]:
raw_dataset["pm25"] = raw_dataset["pm25"].replace("---", -1).astype(int)

ValueError: cannot convert float NaN to integer

In [41]:
raw_dataset["pm25"].isna().sum()

np.int64(62)

In [42]:
raw_dataset["pm25"] = raw_dataset["pm25"].fillna(-1)

# Sanity check the missing values.
raw_dataset["pm25"].isna().sum()

np.int64(0)

In [43]:
raw_dataset["pm25"] = raw_dataset["pm25"].replace("---", -1).astype(int)

In [44]:
aw_dataset["so2"] = raw_dataset["so2"].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [45]:
raw_dataset["so2"] = raw_dataset["so2"].replace("---", -1).astype(int)

In [46]:
raw_dataset["co"] = raw_dataset["co"].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [47]:
raw_dataset["co"] = raw_dataset["co"].replace("---", -1).astype(int)

In [48]:
raw_dataset["o3"] = raw_dataset["o3"].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [49]:
raw_dataset["o3"] = raw_dataset["o3"].replace("---", -1).astype(int)

In [50]:
raw_dataset["no2"] = raw_dataset["no2"].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [51]:
raw_dataset["no2"] = raw_dataset["no2"].replace("---", -1).astype(int)

In [52]:
raw_dataset["max"] = raw_dataset["max"].astype(int)

ValueError: invalid literal for int() with base 10: 'PM25'

In [53]:
raw_dataset[raw_dataset["max"] == "PM25"]

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
757,2021-12-03,DKI1 (Bunderan HI),49,31,9,19,7,49,PM25,BAIK,NaN


In [54]:
raw_dataset.sample(5)

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
107,2021-02-24,DKI4 (Lubang Buaya),42,61,39,9,13,6,61,PM25,SEDANG
1132,2021-05-06,DKI3 (Jagakarsa),39,66,20,9,60,15,66,PM25,SEDANG
954,2021-03-14,DKI2 (Kelapa Gading),50,62,27,12,37,15,62,PM25,SEDANG
712,2021-08-20,DKI4 (Lubang Buaya),64,106,40,11,28,23,106,PM25,TIDAK SEHAT
1410,2021-07-10,DKI2 (Kelapa Gading),78,104,54,10,63,17,104,PM25,TIDAK SEHAT


In [62]:
raw_dataset.loc[757, "max"] = raw_dataset.loc[757, "pm10"]
raw_dataset.loc[757, "critical"] = "PM10"
raw_dataset.loc[757, "categori"] = "BAIK"

In [63]:
raw_dataset.loc[757]

tanggal     2021-12-03 00:00:00
stasiun      DKI1 (Bunderan HI)
pm10                         49
pm25                         31
so2                           9
co                           19
o3                            7
no2                          49
max                          49
critical                   PM10
categori                   BAIK
Name: 757, dtype: object

In [64]:
raw_dataset["max"] = raw_dataset["max"].astype(int)

In [65]:
raw_dataset["critical"].value_counts()

critical
PM25    1630
PM10      66
O3        57
CO        34
SO2       26
Name: count, dtype: int64

In [66]:
missing_labels = raw_dataset[raw_dataset["categori"] == "TIDAK ADA DATA"]
raw_dataset = raw_dataset.drop(index = missing_labels.index)

In [67]:
raw_dataset["categori"].value_counts()

categori
SEDANG         1304
TIDAK SEHAT     319
BAIK            190
Name: count, dtype: int64

In [68]:
raw_dataset = raw_dataset.rename(columns = {"categori": "category"})

In [69]:
config = update_config(
    key = "label",
    value = "category",
    params = config,
    path_config = PATH_CONFIG
)

Params Updated! 
Key: label 
Value: category



In [80]:
col_object = config["object_columns"]
col_object[-1] = "category"

config = update_config(
    key = "columns_object",
    value = col_object,
    params = config,
    path_config = PATH_CONFIG
)

Params Updated! 
Key: columns_object 
Value: ['stasiun', 'critical', 'category']



In [81]:
raw_dataset.info()

<class 'pandas.DataFrame'>
Index: 1813 entries, 0 to 1829
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   tanggal   1813 non-null   datetime64[us]
 1   stasiun   1813 non-null   str           
 2   pm10      1813 non-null   int64         
 3   pm25      1813 non-null   int64         
 4   so2       1813 non-null   int64         
 5   co        1813 non-null   int64         
 6   o3        1813 non-null   int64         
 7   no2       1813 non-null   int64         
 8   max       1813 non-null   int64         
 9   critical  1813 non-null   str           
 10  category  1813 non-null   str           
dtypes: datetime64[us](1), int64(7), str(3)
memory usage: 170.0 KB


In [91]:
config = update_config(
    key = "label",
    value = "category" ,
    params = config,
    path_config = PATH_CONFIG
)

Params Updated! 
Key: label 
Value: category



In [82]:
PATH_VALIDATED_DATA = f"../data/interim/validated_data.pkl"
joblib.dump(raw_dataset, PATH_VALIDATED_DATA)

['../data/interim/validated_data.pkl']

In [83]:
# Update the configuration parameter.
config = update_config(
    key = "path_validated_data",
    value = PATH_VALIDATED_DATA,
    params = config,
    path_config = PATH_CONFIG
)

Params Updated! 
Key: path_validated_data 
Value: ../data/interim/validated_data.pkl



## **4 - Update the Range of Data in Configuration File**
---

In [84]:
cols = ["pm10", "pm25", "so2", "co", "o3", "no2"]
param_keys = ["range_pm10", "range_pm25", "range_so2", "range_co", "range_o3", "range_no2"]

for col, key in zip(cols, param_keys):
    config = update_config(
        key = key,
        value = [int(np.min(raw_dataset[col])), int(np.max(raw_dataset[col]))],
        params = config,
        path_config = PATH_CONFIG
    )

Params Updated! 
Key: range_pm10 
Value: [-1, 179]

Params Updated! 
Key: range_pm25 
Value: [-1, 174]

Params Updated! 
Key: range_so2 
Value: [-1, 82]

Params Updated! 
Key: range_co 
Value: [-1, 47]

Params Updated! 
Key: range_o3 
Value: [-1, 151]

Params Updated! 
Key: range_no2 
Value: [-1, 65]



In [85]:
config

{'columns_object': ['stasiun', 'critical', 'category'],
 'datetime_columns': ['tanggal'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'int32_columns': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'path_raw_data': '../data/raw/',
 'path_validated_data': '../data/interim/validated_data.pkl',
 'range_co': [-1, 47],
 'range_no2': [-1, 65],
 'range_o3': [-1, 151],
 'range_pm10': [-1, 179],
 'range_pm25': [-1, 174],
 'range_so2': [-1, 82],
 'range_stasiun': ['DKI1 (Bunderan HI)',
  'DKI2 (Kelapa Gading)',
  'DKI3 (Jagakarsa)',
  'DKI4 (Lubang Buaya)',
  'DKI5 (Kebon Jeruk) Jakarta Barat']}

## **5 - Data Defense**
---

In [106]:
def check_data(input_data, params):
    """
    Do data defense for checking the data types and range of data.

    Parameters:
    ----------
    input_data : pd.DataFrame
        The data to be checked.

    params : dict
        Loaded configuration parameters.

    Returns:
    -------
    None, it's a void function.
    """

    # Check data types.
    assert input_data.select_dtypes("datetime").columns.to_list() == params["datetime_columns"], "an error occurs in datetime column(s)."
    assert input_data.select_dtypes("str").columns.to_list() == params["columns_object"], "an error occurs in object column(s)."
    assert input_data.select_dtypes("number").columns.to_list() == params["int32_columns"], "an error occurs in int32 column(s)."

    # Check range of data.
    assert set(input_data['stasiun']).issubset(set(params['range_stasiun'])), "an error occurs in stasiun range."
    assert input_data['pm10'].between(params['range_pm10'][0], params['range_pm10'][1]).sum() == len(input_data), "an error occurs in pm10 range."
    assert input_data['pm25'].between(params['range_pm25'][0], params['range_pm25'][1]).sum() == len(input_data), "an error occurs in pm25 range."
    assert input_data['so2'].between(params['range_so2'][0], params['range_so2'][1]).sum() == len(input_data), "an error occurs in so2 range."
    assert input_data['co'].between(params['range_co'][0], params['range_co'][1]).sum() == len(input_data), "an error occurs in co range."
    assert input_data['o3'].between(params['range_o3'][0], params['range_o3'][1]).sum() == len(input_data), "an error occurs in o3 range."
    assert input_data['no2'].between(params['range_no2'][0], params['range_no2'][1]).sum() == len(input_data), "an error occurs in no2 range."

In [107]:
check_data(raw_dataset, config)

## **6 - Data Split**
---

In [95]:
# Input-Output Split.
def split_input_output(input_data, params):
    """
    Split the input(X) and output (y).

    Parameters:
    ----------
    input_data : pd.DataFrame
        The processed dataset.

    params : dict
        Loaded configuration parameters.

    Returns:
    -------
    X : pd.DataFrame
        The input data.

    y : pd.Series
        The output data.
    """

    X = input_data[params["features"]].copy()
    y = input_data[params["label"]].copy()

    print(f"Original data shape : {input_data.shape}")
    print(f"Selected Features   : {params["features"]}")
    print(f"X data shape        : {X.shape}")
    print(f"y data shape        : {y.shape}")

    return X, y

In [96]:
X, y = split_input_output(
    input_data = raw_dataset,
    params = config
)

Original data shape : (1813, 11)
Selected Features   : ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
X data shape        : (1813, 7)
y data shape        : (1813,)


In [97]:
X.head()

,stasiun,pm10,pm25,so2,co,o3,no2
0,DKI1 (Bunderan HI),59,79,21,26,15,31
1,DKI1 (Bunderan HI),30,46,21,14,16,24
2,DKI1 (Bunderan HI),23,33,19,11,14,19
3,DKI1 (Bunderan HI),36,53,23,14,12,24
4,DKI1 (Bunderan HI),29,36,20,14,12,21


In [98]:
y.head()

0    SEDANG
1      BAIK
2      BAIK
3    SEDANG
4      BAIK
Name: category, dtype: str

In [99]:
def split_train_test(X, y, test_size, random_state):
    """
    Split the train and test set.

    Parameters:
    ----------
    X : pd.DataFrame
        The input data.

    y : pd.Series
        The output data.

    test_size : float
        The proportion of test set.

    random_state : int
        For reproducibility

    Returns:
    -------
    X_train, X_test : pd.DataFrame
        The train and test input.

    y_train, y_test : pd.Series
        The train and test output.
    """

    X_train, X_test, y_train, y_test = train_test_split(
                                            X, y,
                                            test_size = test_size,
                                            random_state = random_state,
                                            stratify = y
                                       )

    print(f"X_train shape : {X_train.shape}")
    print(f"y_train shape : {y_train.shape}")
    print(f"X_test shape  : {X_test.shape}")
    print(f"y_test shape  : {y_test.shape}")

    return X_train, X_test, y_train, y_test

In [100]:
# train:valid:test -> 80:10:10

RANDOM_STATE = 123

# Train vs not-train.
X_train, X_not_train, y_train, y_not_train = split_train_test(
    X = X,
    y = y,
    test_size = 0.2,
    random_state = RANDOM_STATE
)

print()
# Valid vs test.
X_valid, X_test, y_valid, y_test = split_train_test(
    X = X_not_train,
    y = y_not_train,
    test_size = 0.5,
    random_state = RANDOM_STATE
)

X_train shape : (1450, 7)
y_train shape : (1450,)
X_test shape  : (363, 7)
y_test shape  : (363,)

X_train shape : (181, 7)
y_train shape : (181,)
X_test shape  : (182, 7)
y_test shape  : (182,)


In [101]:
PATH_SPLITTED_DATA = f"../data/interim/"

joblib.dump(X_train, f"{PATH_SPLITTED_DATA}X_train.pkl")
joblib.dump(y_train, f"{PATH_SPLITTED_DATA}y_train.pkl")
joblib.dump(X_valid, f"{PATH_SPLITTED_DATA}X_valid.pkl")
joblib.dump(y_valid, f"{PATH_SPLITTED_DATA}y_valid.pkl")
joblib.dump(X_test, f"{PATH_SPLITTED_DATA}X_test.pkl")
joblib.dump(y_test, f"{PATH_SPLITTED_DATA}y_test.pkl")

['../data/interim/y_test.pkl']

In [102]:
config = update_config(
    key = "path_train_set",
    value = [f"{PATH_SPLITTED_DATA}X_train.pkl", f"{PATH_SPLITTED_DATA}y_train.pkl"],
    params = config,
    path_config = PATH_CONFIG
)

config = update_config(
    key = "path_valid_set",
    value = [f"{PATH_SPLITTED_DATA}X_valid.pkl", f"{PATH_SPLITTED_DATA}y_valid.pkl"],
    params = config,
    path_config = PATH_CONFIG
)

config = update_config(
    key = "path_test_set",
    value = [f"{PATH_SPLITTED_DATA}X_test.pkl", f"{PATH_SPLITTED_DATA}y_test.pkl"],
    params = config,
    path_config = PATH_CONFIG
)

Params Updated! 
Key: path_train_set 
Value: ['../data/interim/X_train.pkl', '../data/interim/y_train.pkl']

Params Updated! 
Key: path_valid_set 
Value: ['../data/interim/X_valid.pkl', '../data/interim/y_valid.pkl']

Params Updated! 
Key: path_test_set 
Value: ['../data/interim/X_test.pkl', '../data/interim/y_test.pkl']



In [108]:
config

{'columns_object': ['stasiun', 'critical', 'category'],
 'datetime_columns': ['tanggal'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'int32_columns': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'object_columns': ['stasiun', 'critical', 'category'],
 'path_joined_data': '../data/interim/joined_dataset.pkl',
 'path_raw_data': '../data/raw/',
 'path_test_set': ['../data/interim/X_test.pkl', '../data/interim/y_test.pkl'],
 'path_train_set': ['../data/interim/X_train.pkl',
  '../data/interim/y_train.pkl'],
 'path_valid_set': ['../data/interim/X_valid.pkl',
  '../data/interim/y_valid.pkl'],
 'path_validated_data': '../data/interim/validated_data.pkl',
 'range_co': [-1, 47],
 'range_no2': [-1, 65],
 'range_o3': [-1, 151],
 'range_pm10': [-1, 179],
 'range_pm25': [-1, 174],
 'range_so2': [-1, 82],
 'range_stasiun': ['DKI1 (Bunderan HI)